# Unity Catalog Lab: Access Control and Permissions

## Learning Objectives
- Understand Unity Catalog's permission model
- Grant and revoke permissions at each level 
- Test access control with different users/groups
- Use SHOW GRANTS for permission auditing

## Prerequisites
- Completed Labs 1, 2 

## Step 1: Set Up Environment

Let's make sure we're in the right catalog and schema context.

In [0]:
from databricks.sdk import WorkspaceClient

# Get current user context for governance and unique naming
w = WorkspaceClient()
user_name = w.current_user.me().user_name
user_id = w.current_user.me().user_name.split('@')[0]

print(f"Current User: {user_name}")
print(f"User ID for naming: {user_id}")

In [0]:
# Set catalog and schema context for subsequent operations
spark.sql(f"USE CATALOG `workspace`")
spark.sql(f"USE SCHEMA `default`")

In [0]:
%sql
-- Confirm current context
SELECT 
  CURRENT_CATALOG() as current_catalog,
  CURRENT_SCHEMA() as current_schema;

In [0]:
%sql
SHOW TABLES

## Step 2: Granting and Viewing Permissions

Unity Catalog uses a hierarchical permission model. Using commands `GRANT`or `REVOKE` permissions can be inherited from parent objects.

**Permission Hierarchy**:
- **USE_CATALOG**: Access to catalog objects
- **USE_SCHEMA**: Access to schema objects  
- **CREATE_TABLE**: Ability to create tables in schema
- **CREATE_VOLUME**: Ability to create volumes
- **READ_VOLUME/WRITE_VOLUME**: File access permissions
- **SELECT**: Abilityt to select on tables
- all permisions on https://docs.databricks.com/aws/en/data-governance/unity-catalog/manage-privileges/privileges

**Best Practice**: Grant minimal required permissions and use groups instead of individual users.

## Examples
`spark.sql(f"REVOKE SELECT ON TABLE `{table_name}` TO `gonzalo.bautista@bluetab.net` ;")`

## Exercise

#### 1: Grant select on table fct_acitvity and check the grants (using show grants or querying system table `system.information_schema.table_privileges` )

In [0]:
# TODO

## Exercise

#### 2: Revoke select on table fct_acitvity and check the grants 

In [0]:
# TODO

## Step 4: Field masking and Row filtering

Use information schema to programmatically query permissions.

Unity Catalog implements a comprehensive security model through security functions, attribute-based access control (ABAC), and governed tags. These components work together to provide granular data protection.

### Example
filtering rows to specific user/group

In [0]:
spark.sql(f"""
          CREATE OR REPLACE FUNCTION workspace.default.filter_location(location string)
          RETURN IF(current_user()="gonzalobd@outlook.com", location IN ("Endor"), true)
""")

In [0]:
spark.sql(f"""
          ALTER TABLE workspace.default.fct_activity
          SET ROW FILTER workspace.default.filter_location ON (location);
          """)

In [0]:
%sql
SELECT * FROM workspace.default.fct_activity;

## Exercise: 
#### 1.Remove Row filter and query the table again

In [0]:
# TODO


#### 2. Mask function:

- create a function called "mask_activity_name" and set it to your `fct_activity` table that if current user is yourname@mail.com, column `activity_type` shall we viewed as `"XXX activity masked"`

Hint:

`CREATE FUNCTION ... RETURN IF(<condition>, <value>, <column>)`

`ALTER TABLE <>
ALTER COLUMN <> 
SET MASK .... ;`

In [0]:
# TODO

## Lab Summary

**What you learned:**
- ✅ **Permission Model**: Hierarchical structure (Metastore → Catalog → Schema → Object)
- ✅ **Key Permissions**: 
  - `USE CATALOG`, `USE SCHEMA` - Required for access
  - `SELECT`, `READ FILES` - Data operations
  - `CREATE CATALOG`, `CREATE SCHEMA`, `CREATE TABLE` - Object creation
- ✅ **Inheritance**: Permissions can be inherited from parent objects, or granted at each individual level
- ✅ **Auditing**: Use `SHOW GRANTS` and `information_schema` for monitoring
- ✅ **Unity Catalog Security Functions**: Building reusable functions registered in Unity Catalog
- ✅ **ABAC (Attribute-Based Access Control)**: Policy-driven security using user attributes, groups, and context

**Key Commands Used:**
- `GRANT <permission> ON <object> TO <principal>`
- `REVOKE <permission> ON <object> FROM <principal>` 
- `SHOW GRANTS ON <object>`
- `information_schema.table_privileges`
- `information_schema.schema_privileges`
- `CREATE FUNCTION <name> ...`
- `ALTER TABLE <table> SET ROW FILTER <function> ON (<column>)`



**Best Practices:**
- Regularly audit permissions
- Follow principle of least privilege
- Document permission policies for your organization
- Use groups instead of individual user grants
- Use governed tags for consistent data classification
- Implement ABAC policies for scalable security management
- Test security functions with different user contexts
- Document tag taxonomy and policy mappings
- Combine multiple security layers (row, column, object-level)
- Regular review and audit of applied policies


**Troubleshooting Tips:**
- Check all prerequisite permissions for operations
- Verify object existence before granting permissions  
- Use information schema for systematic permission reviews
- Verify user group memberships with `current_groups()`
- Test security functions in different execution contexts
- Validate filter/mask expressions before applying to production tables
